##### Path

In [1]:
result_path = joinpath(@__DIR__,"Result2", "Unfitted_FEM","2D_Cantilever")
isdir(result_path) || mkpath(result_path)

"c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\Result2\\Unfitted_FEM\\2D_Cantilever"

##### Packages

In [2]:
using Gridap
using Gridap.TensorValues
using GridapEmbedded
using GridapTopOpt
using LinearAlgebra

##### Background Mesh

In [3]:
L = 9.0
H = 3.0
nx, ny = 90, 30

domain = (-L/2, L/2, -H/2, H/2)
partition = (nx, ny)
bgmodel = CartesianDiscreteModel(domain,partition)
f_Γ_D(x) = x[1] == -L/2
f_Γ_N(x) = x[1] == L/2 && -H/8 <= x[2] <= H/8
update_labels!(1,bgmodel,f_Γ_D,"Gamma_f_D")
update_labels!(2,bgmodel,f_Γ_N,"Gamma_f_N")

In [4]:
writevtk(bgmodel,joinpath(result_path,"bgmodel"))       

3-element Vector{Vector{String}}:
 ["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\Result2\\Unfitted_FEM\\2D_Cantilever\\bgmodel_0.vtu"]
 ["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\Result2\\Unfitted_FEM\\2D_Cantilever\\bgmodel_1.vtu"]
 ["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\Result2\\Unfitted_FEM\\2D_Cantilever\\bgmodel_2.vtu"]

##### Implicit Cantilever Geometry

##### Implicit Disk

In [5]:
const R = 0.5
const p1 = Point(0.0,0.0)
const p2 = Point(-4.5,-1.5)
e1 = VectorValue(9.0,0.0)
e2 = VectorValue(0.0,3.0)

geo1 = disk(R,x0=p1)

geo2 = quadrilateral(
    x0 = p2,
    d1 = e1,
    d2 = e2
)

geo3 = setdiff(geo2,geo1)

AnalyticalGeometry(Node((:-, "", nothing),Node((:∩, "quadrilateral", nothing),Node((:∩, "", nothing),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{2, Float64}, VectorValue{2, Float64}}((-4.5, -1.5), (0.0, -1.0)), "edge1", nothing)),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{2, Float64}, VectorValue{2, Float64}}((-4.5, 1.5), (-0.0, 1.0)), "edge2", nothing))),Node((:∩, "", nothing),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{2, Float64}, VectorValue{2, Float64}}((-4.5, -1.5), (-1.0, 0.0)), "edge3", nothing)),Leaf((GridapEmbedded.LevelSetCutters.var"#planefun#14"{VectorValue{2, Float64}, VectorValue{2, Float64}}((4.5, -1.5), (1.0, -0.0)), "edge4", nothing)))),Leaf((GridapEmbedded.LevelSetCutters.var"#diskfun#10"{VectorValue{2, Float64}, Float64}((0.0, 0.0), 0.5), "disk", GridapEmbedded.LevelSetCutters.BoundingBox{2, Float64}((-0.505, -0.505), (0.505, 0.505))))))

##### Cut Geometry

In [6]:
# Background Model , Analytical Geometry
cutgeo = cut(bgmodel,geo3)

EmbeddedDiscretization()

In [7]:
Ω_act = Triangulation(cutgeo,ACTIVE)
Ω_bg = Triangulation(bgmodel)
Ω = Triangulation(cutgeo,PHYSICAL)

AppendedTriangulation()

In [8]:
writevtk(Ω_act, joinpath(result_path,"ACTIVE_Triangulation"))
writevtk(Ω_bg, joinpath(result_path,"Background_Triangulation"))
writevtk(Ω, joinpath(result_path,"Physical_Triangulation"))

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\Result2\\Unfitted_FEM\\2D_Cantilever\\Physical_Triangulation.vtu"],)

In [9]:
order = 1
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
Vstd = TestFESpace(Ω_act,reffe,conformity=:H1)

UnconstrainedFESpace()

In [10]:
strategy = AggregateAllCutCells()
aggregates = aggregate(strategy,cutgeo);

In [11]:
colors = color_aggregates(aggregates,bgmodel)
Ω_bg = Triangulation(bgmodel)  
writevtk(Ω_bg,joinpath(result_path,"aggs_on_bg_trian"),celldata=["aggregate"=>aggregates,"color"=>colors])

(["c:\\Users\\IIT BBSR\\Desktop\\Amiya\\BTP\\Unfitted_FEM_Prolems\\Result2\\Unfitted_FEM\\2D_Cantilever\\aggs_on_bg_trian.vtu"],)

In [12]:
V = AgFEMSpace(Vstd,aggregates)
U = TrialFESpace(V)

FESpaceWithLinearConstraints()

In [13]:
# degree = 2*order
# dΩ = Measure(Ω,degree)
# Γ = EmbeddedBoundary(cutgeo)
# n_Γ = get_normal_vector(Γ)
# dΓ = Measure(Γ,degree)

# Γ_N = BoundaryTriangulation(bgmodel,tags="Gamma_f_N")
# dΓ_N = Measure(Γ_N,2)
# Γ_D = BoundaryTriangulation(bgmodel,tags="Gamma_f_D")
# dΓ_D = Measure(Γ_D,2)
# n_Γ_D = get_normal_vector(Γ_D)

In [14]:
writevtk(Γ, joinpath(result_path,"Γ_all"))

LoadError: UndefVarError: `Γ` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
# h = minimum(get_element_diameters(bgmodel)) 
# hₕ = get_element_diameter_field(bgmodel)
# γg = 0.1
h = get_element_diameter_field(bgmodel)
γg = 0.1
γD = 10.0

10.0

In [ ]:
function  ElasFourthOrderConstTensor(E,ν,PlanarState)# 1 for  Plane  Stress  and 2 Plane  Strain  Condition
    if  PlanarState  == 1
        C1111 =E/(1-ν*ν)
        C1122 = (ν*E)/(1-ν*ν)
        C1112 = 0.0
        C2222 =E/(1-ν*ν)
        C2212 = 0.0
        C1212 =E/(2*(1+ν))
    elseif  PlanarState  == 2
        C1111 = (E*(1-ν*ν))/((1+ν)*(1-ν-2*ν*ν))
        C1122 = (ν*E)/(1-ν-2*ν*ν)
        C1112 = 0.0
        C2222 = (E*(1-ν))/(1-ν-2*ν*ν)
        C2212 = 0.0
        C1212 =E/(2*(1+ν))
    end
    C_ten = SymFourthOrderTensorValue(C1111 ,C1112 ,C1122 ,C1112 ,
        C1212 ,C2212 ,C1122 ,C2212 ,C2222)
    return   C_ten
end

ElasFourthOrderConstTensor (generic function with 1 method)

In [ ]:
E = 21000.0
ν = 0.3
const  C = ElasFourthOrderConstTensor(E,ν,2)
g = VectorValue(10,0)
λ = (E*ν)/((1+ν)*(1-2ν))
μ = E/(2*(1+ν))

8076.923076923076

In [ ]:
a(u,v) =
  ∫( ε(v) ⊙ C ⊙ ε(u) )dΩ +
  ∫( γg * h * (λ+2μ) *
     ( jump(n_Γ⋅∇(u)) ⋅ jump(n_Γ⋅∇(v)) )
   )dΓ -
  ∫( ((C ⊙ ε(u)) ⋅ n_Γ_D) ⋅ v )dΓ_D -
  ∫( ((C ⊙ ε(v)) ⋅ n_Γ_D) ⋅ u )dΓ_D +
  ∫( γD * (λ+2μ) / h * (u⋅v) )dΓ_D
l(v) = ∫( v⋅g )dΓ_N

l (generic function with 1 method)

In [ ]:
op = AffineFEOperator(a, l, U, V)
uh = solve(op)

SingleFieldFEFunction():
 num_cells: 2644
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 6214543372969448320

In [ ]:
writevtk(Ω,joinpath(result_path,"results.vtu"),cellfields=["uh"=>uh])  

(["C:\\Users\\Mayank\\AmiyaCodes\\2DCantilever\\Result2\\Unfitted_FEM\\2D_Cantilever\\results.vtu"],)